In [3]:
import os
import torch
import traceback
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline

# === Setup device ===
device = 0 if torch.cuda.is_available() else -1
print(f"Using {'CUDA' if device==0 else 'CPU'}")

# === Load model & tokenizer ===
model_name = "facebook/bart-large-cnn"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
summarizer = pipeline(
    "summarization",
    model=model,
    tokenizer=tokenizer,
    device=device
)

# === Split input into line‐based chunks ===
def split_by_blocks(text, block_size=7):
    lines = text.strip().splitlines()
    return [
        "\n".join(lines[i : i + block_size])
        for i in range(0, len(lines), block_size)
        if lines[i : i + block_size]
    ]

# === Main function ===
def summarize_long_document(
    file_path,
    output_folder="summarized_chunks",
    block_size=7,
    chunk_max_length=150,
    chunk_min_length=40,
    final_max_length=250,
    final_min_length=100
):
    try:
        # อ่านไฟล์ต้นฉบับ
        with open(file_path, "r", encoding="utf-8") as f:
            text = f.read()

        chunks = split_by_blocks(text, block_size)
        print(f"Split into {len(chunks)} chunks")

        os.makedirs(output_folder, exist_ok=True)
        chunk_summaries = []

        # สรุปแต่ละ chunk แล้วเซฟทันที
        for idx, chunk in enumerate(chunks, start=1):
            print(f"[Chunk {idx}/{len(chunks)}] summarizing…")
            result = summarizer(
                chunk,
                max_length=chunk_max_length,
                min_length=chunk_min_length,
                do_sample=False
            )
            summary = result[0]["summary_text"].strip()
            chunk_summaries.append(summary)

            # เซฟไฟล์ chunk
            path = os.path.join(output_folder, f"chunk_{idx:02d}.txt")
            with open(path, "w", encoding="utf-8") as cf:
                cf.write(summary)

        # รวม chunk มาสรุปเป็น final summary
        if len(chunk_summaries) > 1:
            print("Combining chunks into final summary…")
            combined = " ".join(chunk_summaries)
            final = summarizer(
                combined,
                max_length=final_max_length,
                min_length=final_min_length,
                do_sample=False
            )
            final_summary = final[0]["summary_text"].strip()
        else:
            final_summary = chunk_summaries[0]

        # เซฟ final summary
        final_path = os.path.join(output_folder, "final_summary.txt")
        with open(final_path, "w", encoding="utf-8") as ff:
            ff.write(final_summary)

        return final_summary

    except Exception as e:
        traceback.print_exc()
        return f"Error: {e}"

# === Run ===
if __name__ == "__main__":
    result = summarize_long_document("movie.txt")
    print("\n=== Final Summary ===\n")
    print(result)


Using CUDA


Device set to use cuda:0
Your max_length is set to 150, but your input_length is only 79. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=39)


Split into 152 chunks
[Chunk 1/152] summarizing…


Your max_length is set to 150, but your input_length is only 102. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=51)


[Chunk 2/152] summarizing…


...

Your max_length is set to 150, but your input_length is only 63. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=31)


[Chunk 22/152] summarizing…


Your max_length is set to 150, but your input_length is only 44. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=22)


[Chunk 23/152] summarizing…


KeyboardInterrupt: 